# Qwen3-VL v3 event decomposition (prompt r2) — 34-Id regression run

This notebook runs the **v3** (image-conditioned, `{visual, camera, progression}`
schema) event decomposition for exactly the 34 spot-reviewed Ids. The uploaded
bundle contains the v3 code, `train.csv`, the review CSV used as `--ids-file`,
and only the 136 frames of those 34 samples, so it stays a few megabytes.

Resumable state and the final five-column CSV are written directly to Google
Drive. Works from the Colab UI or the VS Code Colab extension (all cells use
plain `subprocess`, no Colab-only widgets). After the run finishes, copy the
Drive output folder back into the local repo at
`preprocessing/generated/qwen3vl_event_chunks_v4_review34/` and run
`python scripts/report_event_chunks_v3_review.py` locally.


In [ ]:
import os
from pathlib import Path

os.environ.setdefault('PYTORCH_CUDA_ALLOC_CONF', 'expandable_segments:True')

DRIVE_ROOT = Path('/content/drive/MyDrive/snu ai')
BUNDLE_PATH = DRIVE_ROOT / 'snu_colab_qwen3vl_event_chunks_v3_bundle.tar'
EXPECTED_BUNDLE_SHA256 = '935463B00BF9DFF66AD59D1F1758ECD3A176332E6115C3C76AE8CD081AC4997C'
WORKDIR = Path('/content/snu_ai')
TRAIN_CSV = WORKDIR / 'snuaichallenge_data/train.csv'
IMAGE_ROOT = WORKDIR / 'snuaichallenge_data/train'
IDS_FILE = (
    WORKDIR
    / 'preprocessing/generated/qwen3vl_event_chunks_900/events_spot_review.csv'
)
OUTPUT_DIR = DRIVE_ROOT / 'preprocessing/qwen3vl_event_chunks_v4_review34'
HF_HOME = Path('/content/huggingface')
os.environ['HF_HOME'] = str(HF_HOME)

SAMPLE_COUNT = 900
SEED = 42
MODEL_ID = 'Qwen/Qwen3-VL-8B-Instruct'
MODEL_REVISION = '0c351dd01ed87e9c1b53cbc748cba10e6187ff3b'
QUANTIZATION = 'auto'
BATCH_SIZE = 0
MAX_NEW_TOKENS = 384
MAX_RETRIES = 1
MAX_PIXELS = 262144
REVIEW_COUNT = 34


## 1. Mount Drive, install dependencies, and extract the bundle

Upload `snu_colab_qwen3vl_event_chunks_v3_bundle.tar` to `MyDrive/snu ai/`
before running this cell.


In [ ]:
import hashlib
import subprocess
import sys

from google.colab import drive

drive.mount('/content/drive')
subprocess.run([
    sys.executable, '-m', 'pip', 'install', '--quiet',
    'transformers==5.13.0', 'accelerate==1.14.0', 'bitsandbytes>=0.46',
], check=True)

if not BUNDLE_PATH.is_file():
    raise FileNotFoundError(
        f'Upload the bundle to this exact path: {BUNDLE_PATH}'
    )
bundle_sha256 = hashlib.sha256(BUNDLE_PATH.read_bytes()).hexdigest().upper()
if bundle_sha256 != EXPECTED_BUNDLE_SHA256:
    raise RuntimeError(
        f'Bundle checksum mismatch: {bundle_sha256} != {EXPECTED_BUNDLE_SHA256}'
    )

WORKDIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
HF_HOME.mkdir(parents=True, exist_ok=True)
subprocess.run(['tar', '-xf', str(BUNDLE_PATH), '-C', str(WORKDIR)], check=True)
if not TRAIN_CSV.is_file():
    raise FileNotFoundError(TRAIN_CSV)
if not IDS_FILE.is_file():
    raise FileNotFoundError(IDS_FILE)
sys.path.insert(0, str(WORKDIR))
print('Bundle verified and extracted:', bundle_sha256)
print('train.csv:', TRAIN_CSV, TRAIN_CSV.stat().st_size, 'bytes')


## 2. CPU preflight: processor accepts `max_pixels` and image messages

Downloads only the small processor/tokenizer files, not the 17.5 GB model.
This fails fast if the pinned transformers build rejects the v3 input format.


In [ ]:
from transformers import AutoProcessor

from preprocessing.qwen3vl_event_chunks import build_messages, load_ids_file

ids = load_ids_file(IDS_FILE)
assert len(ids) == REVIEW_COUNT, len(ids)
frame_paths = sorted(str(path) for path in (IMAGE_ROOT / ids[0]).glob('*.jpg'))
assert len(frame_paths) == 4, frame_paths
processor = AutoProcessor.from_pretrained(
    MODEL_ID, revision=MODEL_REVISION, max_pixels=MAX_PIXELS
)
conversation = build_messages('A preflight caption.', frame_paths)
inputs = processor.apply_chat_template(
    [conversation],
    tokenize=True,
    add_generation_prompt=True,
    return_dict=True,
    return_tensors='pt',
    processor_kwargs={'padding': True},
)
assert 'pixel_values' in inputs and 'image_grid_thw' in inputs, inputs.keys()
print('input_ids shape:', tuple(inputs['input_ids'].shape))
print('image_grid_thw:', inputs['image_grid_thw'].tolist())
print('Preflight passed: max_pixels accepted, four images per message.')


## 3. Prepare and verify the 34-Id sample

CPU-only; deterministically reconstructs the systematic 900 sample, filters it
to the 34 reviewed Ids, and validates that all 136 frames are present.
Re-running is safe and preserves compatible Drive state.


In [ ]:
import csv
import json

prepare_command = [
    sys.executable, '-m', 'preprocessing.qwen3vl_event_chunks',
    '--train-csv', str(TRAIN_CSV),
    '--image-root', str(IMAGE_ROOT),
    '--ids-file', str(IDS_FILE),
    '--output-dir', str(OUTPUT_DIR),
    '--sample-count', str(SAMPLE_COUNT),
    '--seed', str(SEED),
    '--model-id', MODEL_ID,
    '--revision', MODEL_REVISION,
    '--max-new-tokens', str(MAX_NEW_TOKENS),
    '--max-retries', str(MAX_RETRIES),
    '--max-pixels', str(MAX_PIXELS),
    '--prepare-only',
]
subprocess.run(prepare_command, check=True, cwd=WORKDIR)

sample_path = OUTPUT_DIR / 'sampled_captions.csv'
metadata_path = OUTPUT_DIR / 'metadata.json'
with sample_path.open('r', encoding='utf-8', newline='') as file:
    sampled_rows = list(csv.DictReader(file))
assert len(sampled_rows) == REVIEW_COUNT, len(sampled_rows)
metadata = json.loads(metadata_path.read_text(encoding='utf-8'))
assert metadata['validator_version'] == 3, metadata['validator_version']
assert metadata['sample_count'] == REVIEW_COUNT
assert metadata['model']['caption_only'] is False
print('prepared', len(sampled_rows), 'captions for the v3 regression run')


## 4. Inspect the inference GPU and selected loading mode

Inference-only. Full BF16/FP16 weights need at least 22 GiB VRAM; below that
the script selects NF4 4-bit loading. With image inputs the auto batch size is
conservative (2 below 40 GiB).


In [ ]:
import torch

if not torch.cuda.is_available():
    raise RuntimeError('Select a GPU runtime in Colab.')
from preprocessing.qwen3vl_event_chunks import _gpu_configuration

gpu_config = _gpu_configuration(QUANTIZATION, BATCH_SIZE)
print('GPU:', gpu_config['name'])
print(f"VRAM: {gpu_config['total_vram_gib']:.2f} GiB")
print('compute capability:', gpu_config['compute_capability'])
print('dtype:', gpu_config['dtype_name'])
print('quantization:', gpu_config['quantization'])
print('generation batch size:', gpu_config['batch_size'])


## 5. Run or resume the v3 generation for the 34 Ids

The script restores compatible rows from `state.jsonl` first, then generates
only pending Ids. Every Id is recorded: after one correction retry a failed
sample falls back to a single caption-copy event and is logged in
`errors.jsonl`.


In [ ]:
from collections import deque
from datetime import datetime, timezone

run_command = [
    sys.executable, '-m', 'preprocessing.qwen3vl_event_chunks',
    '--train-csv', str(TRAIN_CSV),
    '--image-root', str(IMAGE_ROOT),
    '--ids-file', str(IDS_FILE),
    '--output-dir', str(OUTPUT_DIR),
    '--sample-count', str(SAMPLE_COUNT),
    '--seed', str(SEED),
    '--model-id', MODEL_ID,
    '--revision', MODEL_REVISION,
    '--quantization', QUANTIZATION,
    '--batch-size', str(BATCH_SIZE),
    '--max-new-tokens', str(MAX_NEW_TOKENS),
    '--max-retries', str(MAX_RETRIES),
    '--max-pixels', str(MAX_PIXELS),
]

state_path = OUTPUT_DIR / 'state.jsonl'
errors_path = OUTPUT_DIR / 'errors.jsonl'
fatal_error_path = OUTPUT_DIR / 'fatal_error.log'
log_path = OUTPUT_DIR / 'qwen_run.log'

def nonempty_line_count(path):
    if not path.is_file():
        return 0
    with path.open('r', encoding='utf-8') as file:
        return sum(bool(line.strip()) for line in file)

print('resuming valid state rows:', nonempty_line_count(state_path))
print('previous error records:', nonempty_line_count(errors_path))
print('full streaming log:', log_path)
print('fatal traceback log:', fatal_error_path)

tail = deque(maxlen=120)
with log_path.open('a', encoding='utf-8', buffering=1) as log_file:
    marker = f"\n===== run {datetime.now(timezone.utc).isoformat()} =====\n"
    log_file.write(marker)
    print(marker, end='')
    process = subprocess.Popen(
        run_command,
        cwd=WORKDIR,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
        errors='replace',
    )
    assert process.stdout is not None
    for line in process.stdout:
        print(line, end='')
        log_file.write(line)
        tail.append(line)
    return_code = process.wait()

if return_code != 0:
    print('\n===== INNER ERROR TAIL =====')
    print(''.join(tail), end='')
    raise RuntimeError(
        f'Qwen v3 event generation failed with exit code {return_code}. '
        f'The real traceback is printed above and saved at {log_path} '
        f'and {fatal_error_path}'
    )
print('Qwen v3 event generation completed successfully.')


## 6. Validate the final file and inspect the structured events


In [ ]:
from collections import Counter

import pandas as pd

events_path = OUTPUT_DIR / 'events.csv'
metadata = json.loads(metadata_path.read_text(encoding='utf-8'))
assert metadata['status'] == 'complete', metadata
with events_path.open('r', encoding='utf-8', newline='') as file:
    reader = csv.DictReader(file)
    event_rows = list(reader)
    fieldnames = reader.fieldnames

assert fieldnames == ['Id', 'Sentence', 'Events', 'Camera', 'Progression'], fieldnames
assert len(event_rows) == REVIEW_COUNT
assert [row['Id'] for row in event_rows] == [row['Id'] for row in sampled_rows]

state_records = {}
with (OUTPUT_DIR / 'state.jsonl').open('r', encoding='utf-8') as file:
    for line in file:
        if line.strip():
            payload = json.loads(line)
            state_records[payload['Id']] = payload
fallback_ids = sorted(
    sample_id
    for sample_id, payload in state_records.items()
    if payload.get('fallback')
)
retry_ids = sorted(
    sample_id
    for sample_id, payload in state_records.items()
    if payload.get('attempts', 1) > 1 and not payload.get('fallback')
)

decoded = []
for row in event_rows:
    events = json.loads(row['Events'])
    cameras = json.loads(row['Camera'])
    progressions = json.loads(row['Progression'])
    assert 1 <= len(events) <= 4
    assert len(events) == len(cameras) == len(progressions)
    decoded.append({**row, 'DecodedEvents': events, 'EventCount': len(events)})

print('validated final rows:', len(decoded))
print('event-count distribution:', dict(sorted(Counter(
    row['EventCount'] for row in decoded
).items())))
print('fallback ids:', fallback_ids)
print('retry ids (attempts > 1):', retry_ids)
print('final CSV:', events_path)
review = pd.DataFrame([
    {
        'Id': row['Id'],
        'Sentence': row['Sentence'],
        'Events': '\n'.join(
            f'{index + 1}. [{camera}] {event}'
            + (f' | progression: {progression}' if progression else '')
            for index, (event, camera, progression) in enumerate(zip(
                row['DecodedEvents'],
                json.loads(row['Camera']),
                json.loads(row['Progression']),
            ))
        ),
    }
    for row in decoded
])
display(review)


## Output

The run writes to
`/content/drive/MyDrive/snu ai/preprocessing/qwen3vl_event_chunks_v4_review34/`:
`events.csv`, `state.jsonl`, `errors.jsonl`, `metadata.json`, `qwen_run.log`.

Copy that whole folder back into the local repo as
`preprocessing/generated/qwen3vl_event_chunks_v4_review34/`, then run locally:

```
python scripts/report_event_chunks_v3_review.py
```

The report prints the event-count match table against the review sheet, the
four individually reviewed Ids in full, camera/progression usage, and
fallback/retry counts.
